# Notebook de Entrenamiento — Solo tareas de Sonia

Este notebook contiene **únicamente** lo que corresponde a Sonia en el equipo:

- **Tarea 21**: Crear este notebook.
- **Tarea 24**: Entrenar el Modelo 2 (Perfil Financiero) con `ingreso_mensual` y `nivel_endeudamiento`.
- **Tarea 25**: Evaluar el Modelo 2 con métricas (Accuracy / F1-Score).
- **Tarea 26 (parte de Sonia)**: exportar `perfil_financiero.pkl` con `joblib`.

**Lo que NO está aquí, a propósito:** el procesamiento de texto (TF-IDF, Tarea 22) y el entrenamiento del Modelo 1 - Clasificador de Gastos (Tarea 23) son tareas de **Jacob**. Este notebook no los incluye porque no tenemos su código ni su modelo entrenado. Hay una sección al final marcada como "pendiente de Jacob" para cuando integren su parte.

**Dataset usado**: `dataset_financiero_completo.csv` — tus 3000 personas (550 reales + 2450 simuladas de CDMX), ya generado.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

import joblib

RANDOM_STATE = 42


## 1. Cargar el dataset

In [ ]:
RUTA_DATASET = "../data/processed/dataset_financiero_completo.csv"  # ajusta si tu archivo tiene otro nombre/ruta

df = pd.read_csv(RUTA_DATASET)
print("Filas y columnas:", df.shape)
df.head()


## 2. Exploración y limpieza de datos (EDA)

Solo revisamos las columnas que usa el Modelo 2: `ingreso_mensual`, `nivel_endeudamiento`, `perfil_usuario`.

In [ ]:
df.info()


In [ ]:
df[["ingreso_mensual", "nivel_endeudamiento"]].describe()


In [ ]:
columnas_clave = ["ingreso_mensual", "nivel_endeudamiento", "perfil_usuario"]
filas_antes = len(df)
df = df.dropna(subset=columnas_clave)
print(f"Filas eliminadas por datos faltantes en columnas clave: {filas_antes - len(df)}")


## 3. Preparar datos a nivel usuario

El dataset tiene una fila por transacción; para el Modelo 2 necesitamos un renglón por persona.

In [ ]:
df_usuarios = (
    df.groupby("id_usuario")
      .agg({
          "ingreso_mensual": "first",
          "nivel_endeudamiento": "first",
          "perfil_usuario": "first"
      })
      .reset_index()
)

print("Usuarios únicos:", df_usuarios.shape[0])
df_usuarios["perfil_usuario"].value_counts()


## 4. Modelo 2 — Perfil Financiero (Tarea 24)

Random Forest usando `ingreso_mensual` y `nivel_endeudamiento` para predecir `perfil_usuario`.

In [ ]:
X = df_usuarios[["ingreso_mensual", "nivel_endeudamiento"]]
y = df_usuarios["perfil_usuario"]

codificador_perfil = LabelEncoder()
y_codificado = codificador_perfil.fit_transform(y)
print("Clases:", list(codificador_perfil.classes_))

X_train, X_test, y_train, y_test = train_test_split(
    X, y_codificado, test_size=0.2, random_state=RANDOM_STATE
)

modelo_perfil = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
modelo_perfil.fit(X_train, y_train)

print("Modelo 2 (perfil financiero) entrenado.")


## 5. Evaluación del Modelo 2 (Tarea 25)

In [ ]:
y_pred = modelo_perfil.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Modelo 2 - Perfil financiero")
print(f"Accuracy: {acc:.4f}")
print(f"F1-Score (weighted): {f1:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=codificador_perfil.classes_))


## 6. Guardar el modelo entrenado (Tarea 26 — parte de Sonia)

Se guarda `perfil_financiero.pkl` y el codificador de clases (necesario para traducir la predicción de vuelta a texto).

In [ ]:
joblib.dump(modelo_perfil, "../models/perfil_financiero.pkl")
joblib.dump(codificador_perfil, "../models/codificador_perfil.pkl")

print("Archivos guardados:")
print("- ../models/perfil_financiero.pkl")
print("- ../models/codificador_perfil.pkl")


## 7. Prueba rápida

In [ ]:
modelo_cargado = joblib.load("../models/perfil_financiero.pkl")
codificador_cargado = joblib.load("../models/codificador_perfil.pkl")

usuario_ejemplo = pd.DataFrame({"ingreso_mensual": [4500], "nivel_endeudamiento": [25]})
pred_codificada = modelo_cargado.predict(usuario_ejemplo)
pred = codificador_cargado.inverse_transform(pred_codificada)

print("Usuario: ingreso_mensual=4500, nivel_endeudamiento=25")
print("Perfil financiero predicho:", pred[0])


## Pendiente de Jacob (Tareas 22, 23) — no incluido aquí

Cuando Jacob entregue su parte (vectorización TF-IDF + Modelo 1 entrenado), falta:

1. Que él guarde su modelo como `../models/clasificador_gastos.pkl` y su vectorizador como `../models/vectorizador_gastos.pkl` (mismo patrón que usamos aquí para el Modelo 2).
2. La Tarea 26 completa (que pide **ambos** `.pkl`) queda cerrada hasta que las dos partes estén integradas en el repositorio — cada quien exporta su propio modelo, no hace falta que uno dependa del código del otro.
3. La Tarea 25 completa (evaluar "los modelos", en plural) también se completa cuando se junten las métricas de su Modelo 1 con las del Modelo 2 de aquí arriba.